# Strace: evaluate models and schedulers 


**strace** is a research codebase for **diffusion / masked language models** (e.g. LLaDA, Dream). It wires:

- **`eval.py`** (via **`eval.sh`** or **`launch_eval.py`**) — benchmarks via **lm-eval** (GSM8K, HumanEval, …) with your choice of **sampling / scheduler** (pure diffusion, block diffusion, top‑k, …).
- **`schedulers/`** — implementations of those schedulers; **`test_schedulers.py`** for small smoke tests.
- **Configs** — `config/config.yaml` (Hydra); CLI overrides set task, model path, mode, etc.

**This notebook** focuses on: install → download a model → run a small eval and (optionally) inspect results.

📌 **You need** a [Hugging Face](https://huggingface.co) account and **access token** to download public model weights (and for gated models, you must accept the license on the model page first).

## 1. Get the code

Pick **one** of the following (edit the `git` URL to **your** fork or the canonical remote).

In [ ]:
import os
import subprocess
from pathlib import Path

COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/strace") if COLAB else Path.cwd()

if COLAB and not (REPO_DIR / "eval.py").is_file():
    # Set this to your fork or the upstream URL
    REPO_URL = "https://github.com/devichand579/strace.git"
    subprocess.run(["git", "clone", REPO_URL, "strace"], check=True, cwd="/content")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd().resolve())

## 2. Install Python dependencies

Colab already ships with PyTorch. Installing **the full** `requirements.txt` can work but may be slow or occasionally conflict. We strip **local NVIDIA wheel** lines and install the rest.

If the next cell errors, read the error: often you can `pip install` the package name it names missing.

In [ ]:
import subprocess
import sys
from pathlib import Path

req = Path("requirements.txt")
lines = req.read_text().splitlines()
filtered = [
    ln
    for ln in lines
    if ln.strip()
    and not ln.strip().startswith("#")
    and not ln.split("==")[0].strip().lower().startswith("nvidia-")
]
tmp = Path("/tmp/requirements_colab.txt") if Path("/tmp").is_dir() else Path("requirements_colab_stripped.txt")
tmp.write_text("\n".join(filtered) + "\n")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(tmp)])
print("pip install: ok")

## 3. Hugging Face token (for downloads + datasets)

1. Create a token: [Settings → Access Tokens](https://huggingface.co/settings/tokens) (read is enough for public models).
2. In Colab: **🔑 Secrets** (left sidebar) → add secret name `HF_TOKEN` → paste the token → enable notebook access.
3. Run the next cell. If you are **not** on Colab, set `os.environ["HF_TOKEN"]` yourself or run `huggingface-cli login`.

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # not Colab, or secret not set — use existing env or login interactively

if not os.environ.get("HF_TOKEN"):
    print("Warning: HF_TOKEN not set. Model download may fail. Add Colab secret or export HF_TOKEN.")
else:
    print("HF_TOKEN is set (length %d)" % len(os.environ["HF_TOKEN"]))

## 4. Download a model into `./models/`

Eval expects a folder like `./models/llada` (path passed as `model_path=...` or `--model llada` from scripts). Example: **LLaDA-8B-Base** (large — ~16GB+ on disk; use a smaller or quantized model if you hit disk limits).

See `docs/setup.md` for other models (Dream, LLaDA-1.5).

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_DIR = Path("models/llada")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
if not (MODEL_DIR / "config.json").is_file():
    snapshot_download(
        repo_id="GSAI-ML/LLaDA-8B-Base",
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False,
    )
    print("Downloaded to", MODEL_DIR.resolve())
else:
    print("Model already present:", MODEL_DIR.resolve())

## 5. Put the repo on `PYTHONPATH` and test imports

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
for p in (ROOT, *ROOT.parents):
    if (p / "eval.py").is_file() and (p / "launch_eval.py").is_file():
        ROOT = p
        break
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

## 6. Run a small evaluation 

Use **`launch_eval.py`** (Hydra) so you stay in the **same Python env** as the notebook — no conda activation like **`eval.sh`**. Overrides use `key=value` (see `config/config.yaml` defaults).

The example below matches a typical shell invocation: **`batch_size=4`**, **`mode`** (scheduler shortcut, same as `eval.sh --mode`, e.g. `pd`), **`task_name=gsm8k`**, **`model=llada`** (directory under `./models/`), **`wandb=true`**. Set **`WANDB_API_KEY`** (Colab secret + export, or §7).



In [ ]:
# Same idea as: --batch_size 4 --mode $mode --task_name gsm8k --model llada --wandb True
# tweak `mode` (pd, bd, eb, …) and batch_size; lower batch_size if you OOM

import subprocess
import sys

mode = "pd"

subprocess.check_call(
    [
        sys.executable,
        "launch_eval.py",
        "batch_size=4",
        f"mode={mode}",
        "task_name=gsm8k",
        "model=llada",
        "wandb=true",
    ]
)



## 7. Where to read more

- `docs/evaluation.md` — resumable eval, pre-logit capture, tasks.
- `eval.sh` — optional bash wrapper for clusters (`--slurm` with `sbatch`); requires conda as documented in that script.
- `schedulers/` — scheduler code matching `mode` / `strategy`.